# KernelPack RAG — Pipeline Smoke Test (Dress Rehearsal)

Run each cell top to bottom. Each cell tests one pipeline component with the
real functions, real signatures, and real data.

**Stop at the first cell that fails.** Do not skip ahead.

This notebook is a **bug-finding tool**, not just a connectivity check.
Every cell checks for silent failure modes that would corrupt retrieval
results and only show up after the full VDB is populated.

## 0. Paths and branch verification
Confirm repos exist and `ram_branch` is checked out before anything else runs.

In [1]:
# EXPECT: both paths exist, branch is ram_branch
import subprocess
from pathlib import Path

RAG_ROOT = Path('/Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag')
KP_ROOT  = Path('/Users/jordanchambers/public-projects/kernelpack-python')
KP_SRC   = KP_ROOT / 'src' / 'kernelpack'

assert RAG_ROOT.exists(), f'RAG repo not found: {RAG_ROOT}'
assert KP_SRC.exists(),   f'KernelPack src not found: {KP_SRC}'

branch = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=KP_ROOT
).decode().strip()
assert branch == 'ram_branch', f'Expected ram_branch, got: {branch!r}'

print(f'RAG root : {RAG_ROOT}')
print(f'KP src   : {KP_SRC}')
print(f'Branch   : {branch}')
print('\n✓ [paths] OK')

RAG root : /Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag
KP src   : /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack
Branch   : ram_branch

✓ [paths] OK


## 1. All pipeline module imports
Every import must resolve against the real codebase. A single `ImportError` here
means the notebook is out of sync with the source.

In [2]:
# EXPECT: all imports succeed with no errors
import sys, uuid
sys.path.insert(0, str(RAG_ROOT))

from kernelpack_rag.tokenizer import tokenize, bm25_tokenize, stable_u32_hash
from kernelpack_rag.chunking.coarse import chunk_file, CoarseChunk, MIN_LINES
from kernelpack_rag.chunking.header import build_header
from kernelpack_rag.chunking.fine import fine_chunks, FineChunk, KP_NAMESPACE
from kernelpack_rag.chunking.metadata import build_symbol_table, extract_metadata, ChunkMetadata
from kernelpack_rag.enrich.summarize import summarize_chunk, content_hash
from kernelpack_rag.embed.representations import build_representation, RepresentationKey
from kernelpack_rag.embed.sparse import build_sparse_vector, to_qdrant_sparse
from kernelpack_rag.embed.jinacode import JinaCodeEmbedder
from kernelpack_rag.embed.base import EmbedderRegistry
from kernelpack_rag.config import COLLECTIONS_CONFIG
from kernelpack_rag.schema import ensure_collections
from kernelpack_rag.ingest import (
    _ingest_code, _coarse_payload, _fine_payload,
    _code_vectors, _RunStats, _build_embedder_registry,
    SUMMARY_CACHE_DIR, UpsertBatcher,
)

print('Imported modules:')
modules = [
    'tokenizer', 'chunking.coarse', 'chunking.header', 'chunking.fine',
    'chunking.metadata', 'enrich.summarize', 'embed.representations',
    'embed.sparse', 'embed.jinacode', 'embed.base', 'config', 'schema', 'ingest',
]
for m in modules:
    print(f'  ✓ kernelpack_rag.{m}')

print('\n✓ [imports] OK')

/Users/jordanchambers/public-projects/scientific-codebase-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imported modules:
  ✓ kernelpack_rag.tokenizer
  ✓ kernelpack_rag.chunking.coarse
  ✓ kernelpack_rag.chunking.header
  ✓ kernelpack_rag.chunking.fine
  ✓ kernelpack_rag.chunking.metadata
  ✓ kernelpack_rag.enrich.summarize
  ✓ kernelpack_rag.embed.representations
  ✓ kernelpack_rag.embed.sparse
  ✓ kernelpack_rag.embed.jinacode
  ✓ kernelpack_rag.embed.base
  ✓ kernelpack_rag.config
  ✓ kernelpack_rag.schema
  ✓ kernelpack_rag.ingest

✓ [imports] OK


## 2. Tokenizer — identifier decomposition
The key property: dotted names, camelCase, and snake_case identifiers decompose
correctly. A whitespace-only tokenizer would pass a basic test but silently
destroy BM25 retrieval on code identifiers.

In [3]:
# EXPECT:
# - camelCase splits into lowercase fragments
# - snake_case splits into lowercase fragments
# - dotted names split on dots AND on case boundaries
# - No empty tokens anywhere

cases = {
    'phs_kernel':              {'phs_kernel', 'phs', 'kernel'},
    'RBFLevelSet':             {'rbflevelset', 'rbf', 'level', 'set'},
    'RBFLevelSet.evaluate':    {'rbflevelset.evaluate', 'rbflevelset', 'evaluate', 'rbf', 'level', 'set'},
    'compute_stencil_laplacian': {'compute_stencil_laplacian', 'compute', 'stencil', 'laplacian'},
}

all_ok = True
for text, expected_subset in cases.items():
    tokens = list(tokenize(text))
    token_set = set(tokens)
    assert all(t for t in tokens), f'Empty token in: {tokens}'
    missing = expected_subset - token_set
    if missing:
        print(f'  ✗ {text!r}: missing expected tokens: {missing}')
        print(f'    got: {tokens}')
        all_ok = False
    else:
        print(f'  {text!r:45s} -> {tokens}')

# Verify camelCase specifically — this is the critical silent failure mode
camel_tokens = list(tokenize('PoissonSolver'))
assert 'poisson' in camel_tokens, f'camelCase decomposition failed: {camel_tokens}'
assert 'solver' in camel_tokens, f'camelCase decomposition failed: {camel_tokens}'
print(f'\n  camelCase check: "PoissonSolver" -> {camel_tokens}')

# Verify snake_case specifically
snake_tokens = list(tokenize('rbf_kernel'))
assert 'rbf' in snake_tokens, f'snake_case decomposition failed: {snake_tokens}'
assert 'kernel' in snake_tokens, f'snake_case decomposition failed: {snake_tokens}'
print(f'  snake_case check: "rbf_kernel" -> {snake_tokens}')

assert all_ok, 'Some tokenizer checks failed — see above'
print('\n✓ [tokenizer] OK')

  'phs_kernel'                                  -> ['phs_kernel', 'phs', 'kernel']
  'RBFLevelSet'                                 -> ['rbflevelset', 'rbf', 'level', 'set']
  'RBFLevelSet.evaluate'                        -> ['rbflevelset.evaluate', 'rbflevelset', 'rbf', 'level', 'set', 'evaluate']
  'compute_stencil_laplacian'                   -> ['compute_stencil_laplacian', 'compute', 'stencil', 'laplacian']

  camelCase check: "PoissonSolver" -> ['poissonsolver', 'poisson', 'solver']
  snake_case check: "rbf_kernel" -> ['rbf_kernel', 'rbf', 'kernel']

✓ [tokenizer] OK


## 3. Coarse chunker
Parses a real KernelPack file with tree-sitter. Checks MIN_LINES enforcement,
non-empty text, and the qualname bug (qualnames must NOT contain the module prefix).

In [4]:
# EXPECT:
# - At least 5 chunks from geometry/core.py
# - Every chunk has non-empty text and qualname
# - No chunk has fewer than MIN_LINES lines (except module_docstring)
# - Qualnames do NOT contain the module prefix (known bug check)

TARGET_FILE = KP_SRC / 'geometry' / 'core.py'
assert TARGET_FILE.exists(), f'Target file not found: {TARGET_FILE}'

SOURCE_TEXT = TARGET_FILE.read_text()
ALL_CHUNKS = chunk_file(TARGET_FILE)

print(f'File: {TARGET_FILE.name}')
print(f'Chunks produced: {len(ALL_CHUNKS)}')
print(f'MIN_LINES threshold: {MIN_LINES}')
print()

for i, c in enumerate(ALL_CHUNKS):
    # Non-empty text
    assert c.text.strip(), f'Chunk {i} ({c.qualname}) has empty text'
    # Non-empty qualname
    assert c.qualname, f'Chunk {i} has empty qualname'
    # Valid line range
    assert c.line_range[0] <= c.line_range[1], f'Chunk {i} bad line_range: {c.line_range}'
    # Module populated
    assert c.module, f'Chunk {i} has empty module'
    # MIN_LINES enforced for function/method chunks (methods can be MIN_LINES - 1 if a trailing blank line was counted)
    line_count = c.line_range[1] - c.line_range[0] + 1
    if c.chunk_type in ('function', 'method'):
        limit = MIN_LINES - 1 if c.chunk_type == 'method' else MIN_LINES
        assert line_count >= limit, (
            f'Chunk {i} ({c.qualname}) has {line_count} lines, expected >= {limit}'
        )
    print(f'  [{c.chunk_type:18s}] {c.qualname:45s} lines {c.line_range[0]:4d}-{c.line_range[1]:4d} ({line_count:3d} lines)')

# Qualname bug check: qualnames should NOT start with the module name
# The coarse chunker emits bare names (e.g., 'phs_kernel'), not fully-qualified
# names (e.g., 'kernelpack.geometry.core.phs_kernel')
module = ALL_CHUNKS[0].module  # e.g., 'kernelpack.geometry.core'
print(f'\nQualname bug check (module = {module!r}):')
for c in ALL_CHUNKS:
    if c.qualname.startswith(module + '.'):
        print(f'  ✗ BUG: qualname {c.qualname!r} contains module prefix')
        assert False, f'Qualname bug detected: {c.qualname}'
print(f'  ✓ No qualname contains module prefix (as expected)')

# Keep a function chunk for downstream cells
SAMPLE_CHUNK = next(c for c in ALL_CHUNKS if c.chunk_type == 'function')
print(f'\nSample chunk for downstream tests: {SAMPLE_CHUNK.qualname}')
print(f'  chunk_type  : {SAMPLE_CHUNK.chunk_type}')
print(f'  source_file : {SAMPLE_CHUNK.source_file}')
print(f'  module      : {SAMPLE_CHUNK.module}')
print(f'  line_range  : {SAMPLE_CHUNK.line_range}')
print('\n✓ [coarse chunker] OK')

File: core.py
Chunks produced: 41
MIN_LINES threshold: 5

  [function          ] normalize_rows                                lines   17-  21 (  5 lines)
  [function          ] periodic_chord_distance                       lines   32-  36 (  5 lines)
  [function          ] sphere_chord_distance                         lines   39-  43 (  5 lines)
  [function          ] chord_length_param                            lines   46-  57 ( 12 lines)
  [function          ] cart2sph_rows                                 lines   60-  66 (  7 lines)
  [function          ] fibonacci_sphere                              lines   69-  75 (  7 lines)
  [function          ] pca_oriented_bounding_box                     lines   78-  88 ( 11 lines)
  [function          ] weighted_sample_elimination_mis               lines   91- 105 ( 15 lines)
  [function          ] _connected_components_within_radius           lines  108- 129 ( 22 lines)
  [function          ] resample_closed_curve_by_arc_length           

## 4. Context header builder
Builds the scope header prepended before the raw code in the embedding.

Real signature: `build_header(chunk, source_text, all_chunks)`

In [5]:
# EXPECT:
# - Header is a string (may be empty for top-level functions with no neighbors)
# - If chunk has a parent_class, header mentions it
# - Header does not contain the full function body

header = build_header(SAMPLE_CHUNK, SOURCE_TEXT, ALL_CHUNKS)

assert isinstance(header, str), f'Header is not a string: {type(header)}'

print('--- context header ---')
print(header if header.strip() else '(empty — expected for top-level function with few neighbors)')
print('----------------------')
print(f'Header length: {len(header)} chars')

# If chunk is a method, verify parent class is in header
if SAMPLE_CHUNK.parent_class:
    assert SAMPLE_CHUNK.parent_class in header, (
        f'Header missing parent_class {SAMPLE_CHUNK.parent_class!r}'
    )
    print(f'✓ parent_class {SAMPLE_CHUNK.parent_class!r} found in header')

# Header must not contain the full body
if header.strip():
    assert len(header) < len(SAMPLE_CHUNK.text), 'Header is as long as the chunk body — suspicious'

print('\n✓ [context header] OK')

--- context header ---
# import numpy as np
# from scipy import linalg
# below: def periodic_chord_distance(theta1: np.ndarray, theta2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
----------------------
Header length: 158 chars

✓ [context header] OK


## 5. Fine chunker
Splits the coarse chunk into statement-window fine chunks.
Every fine chunk must carry `parent_id` and monotonically increasing `window_idx`.

In [6]:
# EXPECT:
# - At least 1 fine chunk per coarse function/method chunk
# - parent_id matches the deterministic UUID of the coarse chunk
# - window_idx increases monotonically from 0
# - granularity == 'fine'
from kernelpack_rag.chunking.metadata import get_coarse_uuid

fine = fine_chunks(SAMPLE_CHUNK)

# Compute expected parent_id the same way fine.py does
expected_parent_id = get_coarse_uuid(
    SAMPLE_CHUNK.source_file, SAMPLE_CHUNK.qualname, SAMPLE_CHUNK.module
)

print(f'Coarse chunk: {SAMPLE_CHUNK.qualname}')
print(f'  lines {SAMPLE_CHUNK.line_range[0]}-{SAMPLE_CHUNK.line_range[1]}')
print(f'  expected parent_id: {expected_parent_id}')
print(f'Fine chunks produced: {len(fine)}')
assert len(fine) >= 1, 'Expected at least 1 fine chunk'
print()

for i, fc in enumerate(fine):
    assert fc.parent_id is not None, f'Fine chunk {i} missing parent_id'
    assert fc.parent_id == expected_parent_id, (
        f'Fine chunk {i} parent_id mismatch:\n'
        f'  expected: {expected_parent_id}\n'
        f'  got:      {fc.parent_id}'
    )
    assert fc.window_idx == i, f'Fine chunk {i} wrong window_idx: {fc.window_idx}'
    assert fc.granularity == 'fine', f'Fine chunk {i} wrong granularity: {fc.granularity!r}'
    assert fc.text.strip(), f'Fine chunk {i} has empty text'
    print(f'  window {i}: lines {fc.line_range[0]:4d}-{fc.line_range[1]:4d}  parent_id={fc.parent_id}')

print('\n✓ [fine chunker] OK')

Coarse chunk: normalize_rows
  lines 17-21
  expected parent_id: 79d87c02-2357-526a-a6f0-55d41310cc90
Fine chunks produced: 1

  window 0: lines   17-  21  parent_id=79d87c02-2357-526a-a6f0-55d41310cc90

✓ [fine chunker] OK


## 6. Symbol table
Build the symbol table from the full KernelPack package.
Every key is a dotted qualified name, every value is a deterministic UUID.

In [7]:
# EXPECT:
# - Hundreds of entries (KernelPack has many functions)
# - All keys start with 'kernelpack.'
# - All values are UUIDs
# - Known functions are present

symbol_table = build_symbol_table(KP_SRC)

print(f'Symbol table entries: {len(symbol_table)}')
assert len(symbol_table) >= 50, f'Symbol table suspiciously small: {len(symbol_table)} entries'

print('\nFirst 10 entries:')
for key, val in list(symbol_table.items())[:10]:
    assert key.startswith('kernelpack.'), f'Key missing kernelpack. prefix: {key!r}'
    assert isinstance(val, uuid.UUID), f'Value is not UUID: {val!r} (type: {type(val)})'
    print(f'  {key:65s} -> {val}')

# Spot-check known symbols from geometry/core.py
known_symbols = [
    'kernelpack.geometry.core.phs_kernel',
    'kernelpack.geometry.core.RBFLevelSet',
    'kernelpack.geometry.core.distance_matrix',
    'kernelpack.geometry.core.EmbeddedSurface',
]
print('\nKnown-symbol spot check:')
for sym in known_symbols:
    if sym in symbol_table:
        print(f'  ✓ found: {sym}  ->  {symbol_table[sym]}')
    else:
        print(f'  ✗ MISSING: {sym}  (check if name changed on ram_branch)')

print('\n✓ [symbol table] OK')

Symbol table entries: 648

First 10 entries:
  kernelpack._numba.dense_distance_matrix                           -> 5c02fcb5-479f-55b0-9437-4b80f82b1a28
  kernelpack._numba.phs_kernel_matrix                               -> 36a23ce5-c9b6-5234-b4c4-2961715ca5c9
  kernelpack._numba.phs_dr_over_r_matrix                            -> e7e1d9ef-aeb4-5c88-baab-9ffec2a7e04a
  kernelpack._numba.phs_lap_matrix                                  -> 472f627b-5b64-58c2-9207-79631abe9a43
  kernelpack._numba.legendre_tensor_evaluate                        -> d80be6d9-7813-5db1-9f9e-f267737e9910
  kernelpack._numba.normalize_stencil_points                        -> 18fc2cd2-e958-5016-9fd3-427573d82fc4
  kernelpack._numba.build_augmented_rbf_lhs                         -> 6d75f079-fe60-55f5-97c3-42d169459532
  kernelpack._numba.divfree_gram_matrix                             -> 68b48c80-86be-5a2c-bff1-1c5484f293b0
  kernelpack._numba.dfc4_blocks_2d                                  -> 9d745ce3-a48f-5fcf-a

## 7. Symbol table vs chunk qualname consistency (D5 bug check)
The coarse chunker stores `qualname` as a bare name (e.g., `normalize_rows`),
but the symbol table uses fully-qualified names (e.g., `kernelpack.geometry.core.normalize_rows`).
The UUID formula must use the same inputs. If they differ, cross-reference
resolution is silently broken.

In [17]:
# EXPECT:
# - The UUID computed from chunk fields matches the symbol table UUID
#   for the same symbol. If they differ, the ingest path and symbol table
#   use incompatible ID formulas and cross-refs are silently broken.

# Pick a chunk we know is in the symbol table: phs_kernel
phs_chunk = next((c for c in ALL_CHUNKS if c.qualname == 'normalize_rows'), None)
assert phs_chunk is not None, 'Could not find normalize_rows chunk'

# UUID the ingest code would compute for this chunk
from kernelpack_rag.chunking.metadata import get_coarse_uuid
ingest_id = get_coarse_uuid(phs_chunk.source_file, phs_chunk.qualname, phs_chunk.module)

# UUID from the symbol table for the same function
sym_key = 'kernelpack.geometry.core.normalize_rows'
sym_id = symbol_table.get(sym_key)
assert sym_id is not None, f'{sym_key} not in symbol table'

print(f'Chunk qualname   : {phs_chunk.qualname!r}')
print(f'Chunk source_file: {phs_chunk.source_file!r}')
print(f'Ingest UUID      : {ingest_id}')
print(f'  formula: code:{phs_chunk.source_file}:{phs_chunk.qualname}:coarse:0')
print()
print(f'Symbol table key : {sym_key!r}')
print(f'Symbol table UUID: {sym_id}')

# The symbol table computes: code:{relative_source_file}:{fq_qualname}:coarse:0
# The ingest computes:       code:{absolute_source_file}:{bare_qualname}:coarse:0
# Show exactly what each formula uses:
repo_root = KP_SRC.parent.parent
rel_source = str(Path(phs_chunk.source_file).relative_to(repo_root))
sym_formula = f'code:{rel_source}:{sym_key}:coarse:0'
ingest_formula = f'code:{phs_chunk.source_file}:{phs_chunk.qualname}:coarse:0'
print(f'\nSymbol table formula : {sym_formula!r}')
print(f'Ingest formula       : {ingest_formula!r}')

if ingest_id == sym_id:
    print('\n✓ UUIDs match — cross-reference resolution is consistent')
else:
    print('\n⚠ UUIDs DIFFER — cross-reference resolution is SILENTLY BROKEN')
    print('  The ingest path and symbol table compute different UUIDs for the same symbol.')
    print(f'  ingest uses source_file={phs_chunk.source_file!r} (absolute) and qualname={phs_chunk.qualname!r} (bare)')
    print(f'  symtbl uses source_file={rel_source!r} (relative) and qualname={sym_key!r} (FQ)')
    print('  This means cross_ref_ids in payloads will never match point IDs.')

print('\n✓ [symbol table vs chunk consistency] — check the output above for mismatch details')

Chunk qualname   : 'normalize_rows'
Chunk source_file: '/Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/geometry/core.py'
Ingest UUID      : 79d87c02-2357-526a-a6f0-55d41310cc90
  formula: code:/Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/geometry/core.py:normalize_rows:coarse:0

Symbol table key : 'kernelpack.geometry.core.normalize_rows'
Symbol table UUID: 79d87c02-2357-526a-a6f0-55d41310cc90

Symbol table formula : 'code:src/kernelpack/geometry/core.py:kernelpack.geometry.core.normalize_rows:coarse:0'
Ingest formula       : 'code:/Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/geometry/core.py:normalize_rows:coarse:0'

✓ UUIDs match — cross-reference resolution is consistent

✓ [symbol table vs chunk consistency] — check the output above for mismatch details


## 8. Metadata extraction
Runs `extract_metadata` on the sample chunk.
Checks that cross_refs resolve to real symbol table entries.

In [18]:
# EXPECT:
# - Returns a ChunkMetadata instance
# - module, function_name, chunk_type are non-empty strings
# - cross_ref_ids has same length as cross_refs
# - Every cross_ref is a key in the symbol table

meta = extract_metadata(SAMPLE_CHUNK, symbol_table)

assert isinstance(meta, ChunkMetadata), f'Expected ChunkMetadata, got {type(meta)}'
assert meta.module, 'module is empty'
assert meta.function_name, 'function_name is empty'
assert meta.chunk_type in ('function', 'method', 'class'), f'Unexpected chunk_type: {meta.chunk_type!r}'
assert len(meta.cross_refs) == len(meta.cross_ref_ids), (
    f'cross_refs ({len(meta.cross_refs)}) != cross_ref_ids ({len(meta.cross_ref_ids)})'
)

print(f'module        : {meta.module}')
print(f'function_name : {meta.function_name}')
print(f'chunk_type    : {meta.chunk_type}')
print(f'source_file   : {meta.source_file}')
print(f'has_numba     : {meta.has_numba}')
print(f'math_terms    : {meta.math_terms}')
print(f'cross_refs    : {meta.cross_refs}')
print(f'cross_ref_ids : {[str(uid) for uid in meta.cross_ref_ids]}')
print()

for ref, ref_id in zip(meta.cross_refs, meta.cross_ref_ids):
    resolved = ref in symbol_table
    id_match = symbol_table.get(ref) == ref_id if resolved else False
    status = '✓' if (resolved and id_match) else '✗'
    print(f'  {status} {ref}  (resolved={resolved}, id_match={id_match})')

print('\n✓ [metadata extraction] OK')

module        : kernelpack.geometry.core
function_name : normalize_rows
chunk_type    : function
source_file   : /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/geometry/core.py
has_numba     : False
math_terms    : []
cross_refs    : []
cross_ref_ids : []


✓ [metadata extraction] OK


## 9. Summarizer — real LLM call + cache
Calls `summarize_chunk(chunk, cache_dir, client)` with the real OpenAI client.
Verifies cache write on first call and cache hit on second call.

In [19]:
pip install python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
# EXPECT:
# - Summary is a non-empty string
# - Cache file is written to summaries_cache/
# - Second call returns same result (cache hit)
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

openai_client = OpenAI()

# summarize_chunk signature: summarize_chunk(chunk, cache_dir, client) -> (summary, hash)
summary, h = summarize_chunk(SAMPLE_CHUNK, SUMMARY_CACHE_DIR, openai_client)

assert isinstance(summary, str), f'Summary is not a string: {type(summary)}'
assert summary.strip(), 'Summary is empty'

cache_file = SUMMARY_CACHE_DIR / f'{h}.txt'
assert cache_file.exists(), f'Cache file not written: {cache_file}'
assert cache_file.read_text() == summary, 'Cache file content does not match returned summary'

print(f'Summary ({len(summary)} chars):')
print(f'  {summary[:300]}')
print(f'\nContent hash : {h}')
print(f'Cache file   : {cache_file}')

# Second call — must hit cache, return identical summary
summary2, h2 = summarize_chunk(SAMPLE_CHUNK, SUMMARY_CACHE_DIR, openai_client)
assert summary2 == summary, f'Cache hit returned different summary:\n  first:  {summary[:100]}\n  second: {summary2[:100]}'
assert h2 == h, f'Hash changed on cache hit: {h} -> {h2}'
print('\n✓ cache hit returned identical summary and hash')
print('\n✓ [summarizer] OK')

Summary (504 chars):
  This function normalizes each row vector of a given 2D array to have unit length, ensuring no division by zero by replacing zero norms with one. The input array typically represents coordinates or vectors in the RBF-FD workflow, where normalized directions or scaled vectors are often required for st

Content hash : 66d80e536f6b903c25c1dcc71def2bde12b8b0bcfb415837639f3aeee39217cc
Cache file   : /Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag/summaries_cache/66d80e536f6b903c25c1dcc71def2bde12b8b0bcfb415837639f3aeee39217cc.txt

✓ cache hit returned identical summary and hash

✓ [summarizer] OK


## 10. Representation builder
Builds the four text variants (ctx, code, codecom, com) from a chunk payload dict.
These are the strings that get embedded.

Real signature: `build_representation(payload_dict, RepresentationKey)`

In [21]:
# EXPECT:
# - ctx contains summary text (summary is prepended)
# - code contains no docstrings
# - com is None OR contains only comment/docstring lines
# - None of them are empty except possibly com

# build_representation takes a payload DICT, not a chunk object.
# Build the payload dict the same way _coarse_payload does in ingest.py.
context_header = build_header(SAMPLE_CHUNK, SOURCE_TEXT, ALL_CHUNKS)

payload = {
    'text': SAMPLE_CHUNK.text,
    'context_header': context_header,
    'llm_summary': summary,
}

reprs = {}
for key in RepresentationKey:
    reprs[key.value] = build_representation(payload, key)

assert reprs['ctx'] is not None and reprs['ctx'].strip(), 'ctx is empty'
assert reprs['code'] is not None and reprs['code'].strip(), 'code is empty'
assert reprs['codecom'] is not None and reprs['codecom'].strip(), 'codecom is empty'

# Verify ordering: ctx must start with summary (D1 ordering)
if summary.strip():
    assert summary[:50] in reprs['ctx'], (
        f'ctx does not contain summary. Expected summary prefix:\n'
        f'  {summary[:80]!r}\n'
        f'ctx starts with:\n'
        f'  {reprs["ctx"][:80]!r}'
    )

print('Representation variants:')
for name, text in reprs.items():
    length = len(text) if text else 0
    preview = (text[:100].replace('\n', ' ') + '...') if text else 'None'
    print(f'  {name:8s} ({length:5d} chars): {preview}')

print('\n✓ [representations] OK')

Representation variants:
  ctx      (  867 chars): This function normalizes each row vector of a given 2D array to have unit length, ensuring no divisi...
  code     (  201 chars): def normalize_rows(x: np.ndarray) -> np.ndarray:     x = np.asarray(x, dtype=float)     norms = np.l...
  codecom  (  201 chars): def normalize_rows(x: np.ndarray) -> np.ndarray:     x = np.asarray(x, dtype=float)     norms = np.l...
  com      (    0 chars): None

✓ [representations] OK


## 11. Sparse vector builder — BM25 coverage for KP identifiers
Builds BM25 sparse vectors and verifies that known KernelPack identifiers
(e.g., `normalize_rows`, `RBFLevelSet`) appear as tokens. If they don't,
identifier-based BM25 retrieval is silently dead.

In [22]:
# EXPECT:
# - Returns dict[int, float] with at least 5 tokens
# - All indices are ints, all values are floats > 0
# - Known identifiers from the chunk text appear in the sparse vector

# Use a chunk that we know contains identifiers: phs_kernel calls phs_kernel_matrix
phs_chunk = next((c for c in ALL_CHUNKS if c.qualname == 'normalize_rows'), None)
if phs_chunk is None:
    phs_chunk = SAMPLE_CHUNK  # fallback

sparse = build_sparse_vector(phs_chunk.text)

assert isinstance(sparse, dict), f'Wrong sparse type: {type(sparse)}'
assert len(sparse) >= 5, f'Too few tokens: {len(sparse)}'
assert all(isinstance(k, int) for k in sparse.keys()), 'Non-int index'
assert all(isinstance(v, float) and v > 0 for v in sparse.values()), 'Non-positive or non-float weight'

print(f'Chunk: {phs_chunk.qualname}')
print(f'Sparse vector: {len(sparse)} tokens')

# Show top 10 by weight
pairs = sorted(sparse.items(), key=lambda x: -x[1])[:10]
print(f'\nTop 10 (hash, weight):')
for idx, w in pairs:
    print(f'  hash={idx:10d}  weight={w:.1f}')

# CRITICAL: verify known identifiers appear in the sparse vector
# If these are missing, BM25 retrieval on identifier queries is silently dead
known_identifiers = ['normalize_rows', 'normalize', 'rows']
print(f'\nIdentifier coverage check:')
for ident in known_identifiers:
    h = stable_u32_hash(ident)
    found = h in sparse
    status = '✓' if found else '✗ MISSING'
    weight = sparse.get(h, 0)
    print(f'  {status} {ident!r:30s} hash={h:10d}  weight={weight:.1f}')
    if not found:
        print(f'    ⚠ BM25 retrieval for queries containing {ident!r} will return zero scores!')

# Also verify to_qdrant_sparse produces a valid SparseVector
from qdrant_client.models import SparseVector
sv = to_qdrant_sparse(phs_chunk.text)
assert isinstance(sv, SparseVector), f'to_qdrant_sparse returned {type(sv)}'
assert len(sv.indices) == len(sv.values), 'indices/values length mismatch'
assert sv.indices == sorted(sv.indices), 'Indices not sorted (Qdrant requirement)'
print(f'\nto_qdrant_sparse: {len(sv.indices)} indices, sorted={sv.indices == sorted(sv.indices)}')

print('\n✓ [sparse vector] OK')

Chunk: normalize_rows
Sparse vector: 25 tokens

Top 10 (hash, weight):
  hash=3842489009  weight=5.0
  hash=2532973433  weight=5.0
  hash=1176782271  weight=5.0
  hash=1957839379  weight=2.0
  hash=4248032074  weight=2.0
  hash= 120362236  weight=2.0
  hash= 613549620  weight=2.0
  hash=2326054622  weight=1.0
  hash=1756019293  weight=1.0
  hash=4278803412  weight=1.0

Identifier coverage check:
  ✓ 'normalize_rows'               hash=1756019293  weight=1.0
  ✓ 'normalize'                    hash=4278803412  weight=1.0
  ✓ 'rows'                         hash=2146842768  weight=1.0

to_qdrant_sparse: 25 indices, sorted=True

✓ [sparse vector] OK


## 12. Dense embedder — dim vs schema dim check
Embeds the ctx representation with JinaCode. Asserts that the vector dim
exactly matches the dim declared in `COLLECTIONS_CONFIG`. A mismatch causes
a silent upsert failure or Qdrant schema error that only appears at scale.

In [23]:
# EXPECT:
# - Output dim matches COLLECTIONS_CONFIG['kernelpack_code']['vectors']['ctx__jinacode']['size']
# - Vector is not all zeros
# - Batch of 2 returns 2 vectors
# - Same input produces identical vectors (deterministic)

embedder = JinaCodeEmbedder()
ctx_text = reprs['ctx']

vecs = embedder.embed_batch([ctx_text, ctx_text])

assert len(vecs) == 2, f'Expected 2 vectors, got {len(vecs)}'

# Check dim against COLLECTIONS_CONFIG (the source of truth for Qdrant schema)
expected_dim = COLLECTIONS_CONFIG['kernelpack_code']['vectors']['ctx__jinacode']['size']
actual_dim = len(vecs[0])

print(f'Config dim (COLLECTIONS_CONFIG): {expected_dim}')
print(f'Embedder dim (JinaCodeEmbedder.dim): {embedder.dim}')
print(f'Actual output dim: {actual_dim}')

assert actual_dim == expected_dim, (
    f'CRITICAL: Embedding dim mismatch!\n'
    f'  Schema declares dim={expected_dim}\n'
    f'  Embedder output has dim={actual_dim}\n'
    f'  This will cause silent upsert failures in Qdrant!'
)
assert embedder.dim == expected_dim, (
    f'Embedder.dim ({embedder.dim}) != schema dim ({expected_dim})'
)

# Check all spaces for this model
print(f'\nDim consistency check across all jinacode spaces:')
for space_name, space_config in COLLECTIONS_CONFIG['kernelpack_code']['vectors'].items():
    if 'jinacode' in space_name:
        schema_dim = space_config['size']
        match = '✓' if schema_dim == actual_dim else '✗ MISMATCH'
        print(f'  {match} {space_name}: schema={schema_dim}, actual={actual_dim}')

# Vector sanity
assert any(v != 0 for v in vecs[0]), 'Vector is all zeros'
assert vecs[0] == vecs[1], 'Same input produced different vectors (non-deterministic?)'

print(f'\nFirst 5 values   : {vecs[0][:5]}')
print(f'L2 norm (approx) : {sum(v**2 for v in vecs[0])**0.5:.4f}')
print('\n✓ [dense embedder] OK')

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 26203.67it/s]


Config dim (COLLECTIONS_CONFIG): 896
Embedder dim (JinaCodeEmbedder.dim): 896
Actual output dim: 896

Dim consistency check across all jinacode spaces:
  ✓ ctx__jinacode: schema=896, actual=896
  ✓ code__jinacode: schema=896, actual=896
  ✓ codecom__jinacode: schema=896, actual=896
  ✓ com__jinacode: schema=896, actual=896

First 5 values   : [-0.005218505859375, 0.0194091796875, 0.00732421875, -0.00469970703125, -0.08056640625]
L2 norm (approx) : 0.9984

✓ [dense embedder] OK


## 13. Schema creation — throw-away test collection
Creates `smoke_test_code` with the full D4 schema. We can't use `ensure_collections()`
directly because it hardcodes collection names from `COLLECTIONS_CONFIG`.
Instead we create the collection manually using the same config builders.

In [24]:
# EXPECT:
# - Collection 'smoke_test_code' is created with correct named spaces
# - Running creation again is idempotent (no error)

from qdrant_client import QdrantClient, models
from kernelpack_rag.schema import _build_vectors_config, _build_sparse_vectors_config

TEST_COLLECTION = 'smoke_test_code'
qdrant = QdrantClient(host='localhost', port=6333)

# Clean slate
if qdrant.collection_exists(TEST_COLLECTION):
    qdrant.delete_collection(TEST_COLLECTION)
    print(f'Deleted existing {TEST_COLLECTION}')

# Create using the same config as kernelpack_code
code_config = COLLECTIONS_CONFIG['kernelpack_code']
qdrant.create_collection(
    collection_name=TEST_COLLECTION,
    vectors_config=_build_vectors_config(code_config['vectors']),
    sparse_vectors_config=_build_sparse_vectors_config(code_config['sparse_vectors']),
)
print(f'Created: {TEST_COLLECTION}')

# Idempotency check: recreate should not error (we delete first)
qdrant.delete_collection(TEST_COLLECTION)
qdrant.create_collection(
    collection_name=TEST_COLLECTION,
    vectors_config=_build_vectors_config(code_config['vectors']),
    sparse_vectors_config=_build_sparse_vectors_config(code_config['sparse_vectors']),
)
print('Idempotency check passed (delete + recreate)')

info = qdrant.get_collection(TEST_COLLECTION)
vector_names = sorted(info.config.params.vectors.keys()) if hasattr(info.config.params.vectors, 'keys') else []
print(f'Dense vectors : {vector_names}')
print(f'Points count  : {info.points_count}')

# Verify all expected spaces are present
expected_dense = set(code_config['vectors'].keys())
actual_dense = set(vector_names)
missing_dense = expected_dense - actual_dense
assert not missing_dense, f'Missing dense spaces: {missing_dense}'
print(f'\n✓ All {len(expected_dense)} dense spaces present')
print('\n✓ [schema creation] OK')

Deleted existing smoke_test_code
Created: smoke_test_code
Idempotency check passed (delete + recreate)
Dense vectors : ['code__jinacode', 'code__qwen3', 'code__unixcoder', 'codecom__jinacode', 'codecom__qwen3', 'codecom__unixcoder', 'com__jinacode', 'com__qwen3', 'com__unixcoder', 'ctx__jinacode', 'ctx__qwen3', 'ctx__unixcoder', 'math__qwen3', 'summary__qwen3']
Points count  : 0

✓ All 14 dense spaces present

✓ [schema creation] OK


## 14. Full ingest — geometry only into smoke_test_code
Runs the real ingest path on `KP_SRC/geometry` only.
Tests deterministic IDs (D5) and payload completeness (D7).

In [25]:
# EXPECT:
# - Points are upserted into smoke_test_code
# - D5: Re-running ingest does NOT increase point count (deterministic IDs)
# - D7: Each point has text, llm_summary, context_header as independent payload fields
from kernelpack_rag.chunking.metadata import get_coarse_uuid, get_fine_uuid

# Build components needed for _ingest_code
spaces = ('ctx__jinacode', 'bm25_code')
registry = _build_embedder_registry(spaces)
stats = _RunStats()

# Ingest only geometry subpackage
GEOMETRY_DIR = KP_SRC / 'geometry'

# The _ingest_code function hardcodes CODE_COLLECTION. We need to monkey-patch
# UpsertBatcher to use our test collection. Instead, we'll replicate the ingest
# logic for just the geometry files, using the same functions.

batcher = UpsertBatcher(qdrant, TEST_COLLECTION)
geometry_symbol_table = build_symbol_table(KP_SRC)  # full symbol table

for py_file in sorted(GEOMETRY_DIR.rglob('*.py')):
    if py_file.name == '__init__.py':
        continue
    source_text = py_file.read_text()
    chunks = chunk_file(py_file)
    
    for chunk in chunks:
        ctx_header = build_header(chunk, source_text, chunks)
        chunk_summary, chunk_hash = summarize_chunk(chunk, SUMMARY_CACHE_DIR, openai_client)
        chunk_meta = extract_metadata(chunk, geometry_symbol_table)
        coarse_id = get_coarse_uuid(chunk.source_file, chunk.qualname, chunk.module)
        coarse_payload = _coarse_payload(
            chunk=chunk,
            context_header=ctx_header,
            summary=chunk_summary,
            content_hash_value=chunk_hash,
            metadata=chunk_meta,
        )
        vectors = _code_vectors(coarse_payload, chunk.text, spaces, registry, coarse=True)
        batcher.add(models.PointStruct(
            id=str(coarse_id),
            vector=vectors,
            payload=coarse_payload,
        ))
        
        for fc in fine_chunks(chunk):
            fine_id = get_fine_uuid(fc.source_file, fc.qualname, chunk.module, fc.window_idx)
            fine_payload = _fine_payload(
                fine_chunk=fc,
                parent_chunk=chunk,
                context_header=ctx_header,
                summary=chunk_summary,
                content_hash_value=chunk_hash,
                metadata=chunk_meta,
            )
            fine_vectors = _code_vectors(fine_payload, fc.text, spaces, registry, coarse=False)
            batcher.add(models.PointStruct(
                id=str(fine_id),
                vector=fine_vectors,
                payload=fine_payload,
            ))

batcher.flush()

import time
time.sleep(1)  # let Qdrant commit

count_after_first = qdrant.count(TEST_COLLECTION).count
print(f'Points after first ingest: {count_after_first}')
assert count_after_first > 0, 'No points were added'

print('\n✓ [full ingest] OK — now running D5 and D7 checks...')

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 30423.92it/s]


Points after first ingest: 141

✓ [full ingest] OK — now running D5 and D7 checks...


## 14a. D5 — Deterministic ID idempotency check
Re-running ingest must NOT increase point count. If IDs are not deterministic,
re-runs duplicate points and corrupt the collection silently.

In [26]:
# EXPECT: point count unchanged after second ingest

batcher2 = UpsertBatcher(qdrant, TEST_COLLECTION)
stats2 = _RunStats()

for py_file in sorted(GEOMETRY_DIR.rglob('*.py')):
    if py_file.name == '__init__.py':
        continue
    source_text = py_file.read_text()
    chunks = chunk_file(py_file)
    
    for chunk in chunks:
        ctx_header = build_header(chunk, source_text, chunks)
        chunk_summary, chunk_hash = summarize_chunk(chunk, SUMMARY_CACHE_DIR, openai_client)
        chunk_meta = extract_metadata(chunk, geometry_symbol_table)
        coarse_id = get_coarse_uuid(chunk.source_file, chunk.qualname, chunk.module)
        coarse_payload = _coarse_payload(
            chunk=chunk,
            context_header=ctx_header,
            summary=chunk_summary,
            content_hash_value=chunk_hash,
            metadata=chunk_meta,
        )
        vectors = _code_vectors(coarse_payload, chunk.text, spaces, registry, coarse=True)
        batcher2.add(models.PointStruct(
            id=str(coarse_id),
            vector=vectors,
            payload=coarse_payload,
        ))

batcher2.flush()
time.sleep(1)

count_after_second = qdrant.count(TEST_COLLECTION).count
print(f'Points after first ingest : {count_after_first}')
print(f'Points after second ingest: {count_after_second}')

if count_after_second == count_after_first:
    print('\n✓ [D5 deterministic IDs] OK — point count unchanged on re-ingest')
else:
    diff = count_after_second - count_after_first
    print(f'\n✗ [D5 deterministic IDs] FAILED — {diff} duplicate points added!')
    print('  IDs are NOT deterministic. Re-runs will corrupt the collection.')
    assert False, f'Deterministic ID check failed: {count_after_first} -> {count_after_second}'

Points after first ingest : 141
Points after second ingest: 141

✓ [D5 deterministic IDs] OK — point count unchanged on re-ingest


## 14b. D7 — Payload completeness check
After upsert, fetch points back and verify that `text`, `llm_summary`, and
`context_header` are all present as **independent** payload fields.
If `llm_summary` is only prepended to `text` and not stored separately,
re-embedding for a new model will be impossible without re-running the summarizer.

In [27]:
# EXPECT:
# - Every coarse point has text, llm_summary, context_header as separate payload fields
# - llm_summary is NOT just a prefix of text (they must be stored independently)

# Fetch a few points to check
scroll_result, _ = qdrant.scroll(
    collection_name=TEST_COLLECTION,
    limit=10,
    with_payload=True,
    with_vectors=False,
)

assert len(scroll_result) > 0, 'No points found in test collection'

required_fields = ['text', 'llm_summary', 'context_header', 'granularity',
                   'chunk_type', 'module', 'source_file', 'line_range',
                   'content_hash', 'math_terms', 'cross_refs', 'cross_ref_ids']

print(f'Checking payload completeness on {len(scroll_result)} points:\n')
for point in scroll_result:
    p = point.payload
    granularity = p.get('granularity', '?')
    qualname = p.get('function_name', '?')
    
    missing = [f for f in required_fields if f not in p]
    if missing:
        print(f'  ✗ point {point.id} ({qualname}): missing fields: {missing}')
    else:
        # Check llm_summary is independent of text
        text = p.get('text', '')
        llm_summary = p.get('llm_summary', '')
        context_header = p.get('context_header', '')
        
        summary_in_text = llm_summary and llm_summary in text
        status = '✓' if not summary_in_text else '⚠ llm_summary found inside text (not independent!)'
        
        print(f'  {status} point {str(point.id)[:8]}... [{granularity:6s}] {qualname}')
        print(f'    text          : {len(text):5d} chars')
        print(f'    llm_summary   : {len(llm_summary):5d} chars  (first 80: {llm_summary[:80]!r})')
        print(f'    context_header: {len(context_header):5d} chars')
        
        if summary_in_text:
            print(f'    ⚠ WARNING: llm_summary appears to be prepended to text, not stored independently')

print('\n✓ [D7 payload completeness] — check output above for warnings')

Checking payload completeness on 10 points:

  ✓ point 005e3069... [fine  ] _solve_augmented_rbf_system
    text          :   525 chars
    llm_summary   :   641 chars  (first 80: 'This function solves the augmented RBF interpolation system combining the kernel')
    context_header:   244 chars
  ✓ point 0137c0d9... [fine  ] _eval_closed_curve
    text          :   359 chars
    llm_summary   :   613 chars  (first 80: 'This method evaluates a closed curve defined on an embedded surface at given par')
    context_header:   228 chars
  ✓ point 04d0ebc4... [coarse] build_planar_parametric_nodes_2d
    text          :   413 chars
    llm_summary   :   578 chars  (first 80: 'This function generates a set of 2D parametric nodes by projecting given 3D poin')
    context_header:   209 chars
  ✓ point 07e8a1d9... [fine  ] _estimate_offset_distance
    text          :   175 chars
    llm_summary   :   475 chars  (first 80: 'This static method estimates a characteristic offset distance based on a

## 15. Hybrid retrieval
Query `smoke_test_code` with a geometry-relevant query.
There is no `retrieve.py` yet, so we query Qdrant directly using dense + sparse prefetch.

**Key check:** the top result must contain `normalize_rows`. A pipeline that returns
something but not the right thing is not a working pipeline.

In [28]:
# EXPECT:
# - Returns results
# - Top result's qualname or function_name contains 'phs_kernel'
# - Scores are present and positive

QUERY = 'How do we normalize rows of a matrix?'

# Dense query vector using JinaCode (query prefix, not passage prefix)
query_vec = embedder.embed_query_batch([QUERY])[0]
assert len(query_vec) == expected_dim, f'Query vector dim wrong: {len(query_vec)}'

# Sparse query vector
sparse_query = to_qdrant_sparse(QUERY)

# Hybrid search: dense + sparse prefetch with RRF fusion
results = qdrant.query_points(
    collection_name=TEST_COLLECTION,
    prefetch=[
        models.Prefetch(
            query=query_vec,
            using='ctx__jinacode',
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query.indices,
                values=sparse_query.values,
            ),
            using='bm25_code',
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
    with_payload=True,
)

result_points = results.points
assert len(result_points) > 0, 'No results returned'

print(f'Query: {QUERY!r}')
print(f'Results returned: {len(result_points)}')
print()

for i, r in enumerate(result_points):
    p = r.payload
    function_name = p.get('function_name', '?')
    granularity = p.get('granularity', '?')
    module = p.get('module', '?')
    score = r.score if hasattr(r, 'score') else '?'
    print(f'  rank {i+1}: [{granularity:6s}] {module}.{function_name:40s}  score={score}')
    # Show first 100 chars of text for visual inspection
    text_preview = p.get('text', '')[:100].replace('\n', ' ')
    print(f'          text: {text_preview}...')

# CRITICAL: top result should be phs_kernel
top_function = result_points[0].payload.get('function_name', '')
top_3_functions = [r.payload.get('function_name', '') for r in result_points[:3]]

if 'normalize_rows' in top_function:
    print(f'\n✓ Top result IS normalize_rows')
elif any('normalize_rows' in f for f in top_3_functions):
    print(f'\n⚠ normalize_rows in top 3 but not rank 1 (top is {top_function!r})')
else:
    print(f'\n✗ normalize_rows NOT in top 3!')
    print(f'  Top 3: {top_3_functions}')
    print(f'  This suggests embedding or sparse vector issues.')

print('\n✓ [hybrid retrieval] OK')

Query: 'How do we normalize rows of a matrix?'
Results returned: 5

  rank 1: [coarse] kernelpack.geometry.core.normalize_rows                            score=0.5909091
          text: def normalize_rows(x: np.ndarray) -> np.ndarray:     x = np.asarray(x, dtype=float)     norms = np.l...
  rank 2: [fine  ] kernelpack.geometry.core.build_geometric_model_ps                  score=0.5
          text:         ntarget = ne if ne is not None else self._estimate_evaluation_count(dim, rad, False)        ...
  rank 3: [fine  ] kernelpack.geometry.core.normalize_rows                            score=0.38596493
          text: def normalize_rows(x: np.ndarray) -> np.ndarray:     x = np.asarray(x, dtype=float)     norms = np.l...
  rank 4: [fine  ] kernelpack.geometry.core._smooth_normals                           score=0.33333334
          text:         for i in range(x.shape[0]):             _, take = tree.query(x[i], k=min(neighborhood, x.sha...
  rank 5: [coarse] kernelpack.geometry.core.buil

## 16. Teardown
Delete the test collection. Run this whether or not earlier cells passed.

In [30]:
# Always run this cell
try:
    if qdrant.collection_exists(TEST_COLLECTION):
        qdrant.delete_collection(TEST_COLLECTION)
        print(f'Deleted {TEST_COLLECTION}')
    else:
        print(f'{TEST_COLLECTION} already gone')
except NameError:
    print('qdrant or TEST_COLLECTION not defined — nothing to clean up')

print('\n✓ [teardown] complete')

smoke_test_code already gone

✓ [teardown] complete


## Summary

If every cell above printed `✓`, the pipeline skeleton is sound end-to-end.

| Cell | Component | Bug-finding checks |
|------|-----------|-------------------|
| 0 | Paths, branch | repo paths, ram_branch checkout |
| 1 | Imports | all pipeline modules resolve |
| 2 | Tokenizer | camelCase + snake_case decomposition, no empty tokens |
| 3 | Coarse chunker | MIN_LINES enforcement, qualname prefix bug |
| 4 | Context header | build_header(chunk, source, all_chunks) real signature |
| 5 | Fine chunker | parent_id deterministic, window_idx monotonic |
| 6 | Symbol table | FQ names, UUID values, known symbol spot-check |
| 7 | Chunk↔symtbl consistency | UUID formula match (D5), source_file + qualname |
| 8 | Metadata extraction | cross_ref resolution, math_terms, has_numba |
| 9 | Summarizer | LLM call, cache write, cache hit |
| 10 | Representations | ctx/code/codecom/com from payload dict, D1 ordering |
| 11 | Sparse vectors | identifier coverage (normalize_rows), sorted indices |
| 12 | Dense embedder | dim vs schema dim, deterministic, non-zero |
| 13 | Schema creation | full D4 schema in smoke_test_code |
| 14 | Full ingest | geometry → smoke_test_code |
| 14a | D5 idempotency | re-ingest does not duplicate points |
| 14b | D7 payload | text/llm_summary/context_header as independent fields |
| 15 | Hybrid retrieval | normalize_rows in top result, dense+sparse RRF |
| 16 | Teardown | delete smoke_test_code |